Kütüphane Kurulumu ve GPU Kontrolü

In [1]:
!pip install -q ultralytics

import torch
import os
import glob
import shutil
import random
from ultralytics.data.utils import check_det_dataset
print("Kullanılan Cihaz:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (Lütfen T4 GPU seçin!)")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Kullanılan Cihaz: Tesla T4


Veri Setini İndirme ve Bölme

In [2]:
# Pascal VOC veri setini indir
data_info = check_det_dataset("VOC.yaml")
voc_root = str(data_info["path"])

# Tüm görüntüleri toplayıp sabit seed ile karıştır
all_images = glob.glob(f"{voc_root}/images/*/*.jpg")
random.seed(42)
random.shuffle(all_images)

# Görüntüleri %70 %15 %15 şeklinde böl
total = len(all_images)
train_end = int(total * 0.70)
val_end = int(total * 0.85)

splits = {
    "train": all_images[:train_end],
    "val": all_images[train_end:val_end],
    "test": all_images[val_end:]
}

#Fotoğraf yollarını yeni bir klasöre kopyalar ve her fotoğrafı kendi etiket (.txt) dosyasıyla eşleştirir.
custom_root = "/content/datasets/VOC_Split"
for split_name, img_paths in splits.items():
    os.makedirs(f"{custom_root}/images/{split_name}", exist_ok=True)
    os.makedirs(f"{custom_root}/labels/{split_name}", exist_ok=True)

    for img_path in img_paths:
        label_path = img_path.replace("/images/", "/labels/").replace(".jpg", ".txt")
        if os.path.exists(label_path):
            shutil.copy(img_path, f"{custom_root}/images/{split_name}/")
            shutil.copy(label_path, f"{custom_root}/labels/{split_name}/")

print(f"Toplam {total} görüntü başarıyla bölündü:")
print(f" - Eğitim (Train %70): {len(os.listdir(f'{custom_root}/images/train'))}")
print(f" - Doğrulama (Val %15): {len(os.listdir(f'{custom_root}/images/val'))}")
print(f" - Test (Test %15): {len(os.listdir(f'{custom_root}/images/test'))}")


WARNING ⚠️ Dataset 'VOC.yaml' images not found, missing path '/content/datasets/VOC/images/test2007'
Unzipping /content/datasets/VOC/images/VOCtest_06-Nov-2007.zip to /content/datasets/VOC/images/VOCdevkit...: 100% ━━━━━━━━━━━━ 10357/10357 433.1files/s 23.9s
Unzipping /content/datasets/VOC/images/VOCtrainval_06-Nov-2007.zip to /content/datasets/VOC/images/VOCdevkit...: 100% ━━━━━━━━━━━━ 10945/10945 461.1files/s 23.7s
Unzipping /content/datasets/VOC/images/VOCtrainval_11-May-2012.zip to /content/datasets/VOC/images/VOCdevkit...: 100% ━━━━━━━━━━━━ 40189/40189 1.4Kfiles/s 28.5s
train2012: 100% ━━━━━━━━━━━━ 5717/5717 3.2Kit/s 1.8s
val2012: 100% ━━━━━━━━━━━━ 5823/5823 3.1Kit/s 1.9s
train2007: 100% ━━━━━━━━━━━━ 2501/2501 3.2Kit/s 0.8s
val2007: 100% ━━━━━━━━━━━━ 2510/2510 3.2Kit/s 0.8s
test2007: 100% ━━━━━━━━━━━━ 4952/4952 2.4Kit/s 2.0s
Dataset download success ✅ (58.7s), saved to /content/datasets

Toplam 21503 görüntü başarıyla bölündü:
 - Eğitim (Train %70): 15052
 - Doğrulama (Val %15): 

YAML Yapılandırma Dosyası

In [3]:
import yaml

voc_classes = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat",
    "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]

yaml_data = {
    "path": "/content/datasets/VOC_Split",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 20,
    "names": voc_classes
}

with open("/content/voc_custom.yaml", "w") as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print("'/content/voc_custom.yaml' dosyası oluşturuldu!")

'/content/voc_custom.yaml' dosyası oluşturuldu!


Model Eğitimi

In [4]:
import time
from ultralytics import YOLO

# Hocanın örneğindeki gibi güncel YOLO11 mimarisinin Small (s) versiyonu
model = YOLO("yolo11s.pt")

baslangic = time.time()

results = model.train(
    data="/content/voc_custom.yaml",
    epochs=20,
    imgsz=640,
    batch=32,
    optimizer="AdamW",
    lr0=0.001,
    patience=8,
    save_period=5,
    seed=123,
    project="/content/runs/detect",
    name="voc_yolo11s"
)

bitis = time.time()
toplam_dk = (bitis - baslangic) / 60
print(f"\nEğitim Tamamlandı! Toplam Eğitim Süresi: {toplam_dk:.2f} dakika")

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/voc_custom.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=voc_yolo11s, nbs=64, nms=None, opset=None, optimize=False, opti

Rapor Metrikleri

In [5]:
best_model = YOLO("/content/runs/detect/voc_yolo11s/weights/best.pt")

metrics = best_model.val(data="/content/voc_custom.yaml", split="test")

print("\n--- TEST KÜMESİ PERFORMANS SONUÇLARI ---")
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")
print(f"Tahmin Süresi (Inference): {metrics.speed['inference']:.2f} ms / görüntü")

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,420,540 parameters, 0 gradients, 21.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 40.1±24.5 MB/s, size: 108.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/datasets/VOC_Split/labels/test... 3226 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3226/3226 489.6it/s 6.6s
val: New cache created: /content/datasets/VOC_Split/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 202/202 5.3it/s 37.8s
                   all       3226       7752      0.825      0.737      0.816      0.627
             aeroplane        187        244      0.919      0.828       0.88      0.707
               bicycle        149        186      0.879      0.779      0.868 

Rapor Grafiklerini İndirme

In [6]:
!zip -r /content/proje_ciktilari.zip /content/runs/detect/voc_yolo11s /content/voc_custom.yaml
files.download("/content/proje_ciktilari.zip")

  adding: content/runs/detect/voc_yolo11s/ (stored 0%)
  adding: content/runs/detect/voc_yolo11s/val_batch0_pred.jpg (deflated 13%)
  adding: content/runs/detect/voc_yolo11s/results.png (deflated 7%)
  adding: content/runs/detect/voc_yolo11s/results.csv (deflated 58%)
  adding: content/runs/detect/voc_yolo11s/confusion_matrix.png (deflated 13%)
  adding: content/runs/detect/voc_yolo11s/train_batch4712.jpg (deflated 12%)
  adding: content/runs/detect/voc_yolo11s/BoxF1_curve.png (deflated 3%)
  adding: content/runs/detect/voc_yolo11s/val_batch2_labels.jpg (deflated 9%)
  adding: content/runs/detect/voc_yolo11s/train_batch4711.jpg (deflated 13%)
  adding: content/runs/detect/voc_yolo11s/val_batch0_labels.jpg (deflated 13%)
  adding: content/runs/detect/voc_yolo11s/BoxP_curve.png (deflated 5%)
  adding: content/runs/detect/voc_yolo11s/weights/ (stored 0%)
  adding: content/runs/detect/voc_yolo11s/weights/last.pt (deflated 8%)
  adding: content/runs/detect/voc_yolo11s/weights/best.pt (defla

NameError: name 'files' is not defined

In [7]:
from google.colab import files
files.download("/content/proje_ciktilari.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Gradio Web Uygulaması

In [9]:
import gradio as gr

app_model = YOLO("/content/runs/detect/voc_yolo11s/weights/best.pt")

def nesne_tespit_et(goruntu, guven_esigi):
    if goruntu is None:
        return None, "Lütfen bir görüntü yükleyin."

    sonuclar = app_model.predict(source=goruntu, conf=guven_esigi, imgsz=640)
    sonuc = sonuclar[0]

    # Kutucuk çizilmiş görüntüyü BGR formatından RGB formatına çevir
    cizimli_goruntu = sonuc.plot()[:, :, ::-1]

    tespitler = []
    for kutu in sonuc.boxes:
        sinif_id = int(kutu.cls[0])
        sinif_adi = app_model.names[sinif_id]
        skor = float(kutu.conf[0])
        tespitler.append(f"- {sinif_adi}: %{skor*100:.1f}")

    ozet = "\n".join(tespitler) if tespitler else "Belirlenen eşik değerinde nesne tespit edilemedi."
    return cizimli_goruntu, ozet

arayuz = gr.Interface(
    fn=nesne_tespit_et,
    inputs=[
        gr.Image(type="pil", label="Fotoğraf Yükleyin"),
        gr.Slider(minimum=0.1, maximum=0.9, value=0.40, step=0.05, label="Güven Eşiği (Confidence Threshold)")
    ],
    outputs=[
        gr.Image(type="numpy", label="Tespit Sonucu"),
        gr.Textbox(label="Tespit Edilen Nesneler ve Güven Skorları", lines=8)
    ],
    title="YOLO11s Nesne Tespiti Uygulaması (PASCAL VOC)",
    description="Bir fotoğraf yükleyin; model 20 farklı nesne sınıfını tespit edip konumlarını işaretleyecektir."
)

arayuz.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://907cf6b351e7de26c5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
